In [ ]:
# What: Configure environment, clone repository if running in Colab, and set project root.
# Expected outcome: Project root path added to sys.path with dependencies installed.
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo_dir = Path('/content/USD-Waymo-Capstone-Project')
    if not repo_dir.exists():
        !git clone https://github.com/bartteeuwen/USD-Waymo-Capstone-Project.git {repo_dir}
    os.chdir(repo_dir)
    !pip install -q -r requirements.txt
    !pip install -q waymo-open-dataset-tf-2-12-0 --no-deps

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

# Turning Autonomous Driving Motion Data into Interpretable Scene Intelligence
**Executive White Paper & Analytical Diagnostic Framework**  
*Bart Sosa-Teeuwen | University of San Diego, Master of Science in Applied Data Science*

## Abstract
Autonomous-vehicle fleets generate more trajectory data than safety teams can efficiently inspect. This project converts Waymo Open Motion Dataset scene graphs into interpretable map-friction, motion, and spatial-proximity features for post-drive scenario triage. Random Forest, XGBoost, LightGBM, and graph-model baselines are evaluated against a kinematic-risk target; the selected lightweight model supports fast, explainable review. The companion Streamlit dashboard turns the analysis into a human-in-the-loop safety-screening workflow.

---
## Executive Overview: Turning Driving Data into Actionable Insights

### 1. The Challenge
Self-driving vehicle fleets capture massive streams of movement data every second. Manually reviewing thousands of hours of ordinary driving to find a handful of tricky situations is too slow and expensive for safety teams. This project builds a smart filter that automatically weeds out routine driving and flags complex, high-risk scenes for human safety experts to inspect.

### 2. Choosing a Fast, Transparent Model
Several artificial intelligence models were tested, including complex neural networks. While one complex graph model was slightly more accurate on paper, a **tree-based model (Random Forest)** was selected for real-world use because it delivers:
* **Split-Second Speed:** Scores an entire driving scene in less than **15 milliseconds**.
* **Clear Explanations:** Shows exactly *why* a drive was flagged (like sudden braking or tight spacing between cars) rather than hiding behind confusing math.
* **Balanced Accuracy:** Consistently catches real risks without overwhelming teams with false alarms.

### 3. Business Impact & Efficiency Quantified
When tested on a batch of **6,311 driving scenarios**, the system proved its value immediately:
* **89.8% Automated Filtering:** Automatically identified safe, routine drives and bypassed them.
* **283.5 Hours Saved:** Allowed safety experts to focus exclusively on the top 10% most complex scenes.
* **$24,000+ Saved Per Batch:** Cut review costs dramatically (saving ~$24,097 in engineering time per test batch) while keeping safety high.

### 4. Built for Human Reviewers
* **Easy-to-Use Web Dashboard:** Safety engineers can open an interactive web app to view flagged drives, adjust risk sensitivity, and watch visual video playbacks of vehicle paths.
* **Clean, Modular Code:** Organized into simple, modular pieces so any developer or team can easily run the system on their own computer or cloud storage.
---

## Table of Contents
1. [Business Background](#business-background)  
2. [Problem Statement](#problem-statement)  
3. [Summary of Findings](#summary-of-findings)  
4. [Business Questions, Scope, and Approach](#questions-scope-approach)  
5. [Data Preparation and Quality Screening](#data-preparation)  
6. [Risk Modeling and Fine-Tuning](#risk-modeling)  
7. [Limitations](#limitations)  
8. [Solution Details](#solution-details)  
9. [Concluding Summary and Call to Action](#concluding-summary)

<a id="business-background"></a>
## 1. Business Background
Autonomous-vehicle safety teams must search large collections of post-drive telemetry to find the relatively small number of scenes worth review. Raw Waymo motion records are nested protobuf objects and are difficult to audit directly. A reliable triage layer can reduce routine review work while preserving a transparent record of why a scene was flagged.

[Back to contents](#table-of-contents)

<a id="problem-statement"></a>
## 2. Problem Statement
Safety engineers need a lightweight, interpretable way to prioritize complex multi-agent driving scenes without depending exclusively on expensive end-to-end deep-learning re-simulation. The solution must transform raw trajectory and map data into auditable features, score each scene consistently, and present the evidence in a format suitable for human review.

[Back to contents](#table-of-contents)

<a id="summary-of-findings"></a>
## 3. Summary of Findings
The analysis shows that dynamic interaction features, especially velocity variation and minimum inter-agent distance, add useful risk signal beyond simple actor and road-feature counts. The Graph Attention Network achieved the strongest reported accuracy, but the Random Forest was selected for the operational workflow because it provides competitive discrimination, fast inference, and clear feature-level explanations. Fine-tuning is reported separately to distinguish gains from feature engineering from gains due to hyperparameter search.

[Back to contents](#table-of-contents)

<a id="questions-scope-approach"></a>
## 4. Business Questions, Scope, and Approach
**Business questions.** Can engineered trajectory and map features identify high-complexity scenes? Which interpretable features contribute most to the risk score? Can a lightweight model support a practical safety-review workflow?

**Scope.** The analysis covers Waymo Open Motion Dataset scenarios, map and track feature extraction, physics screening, kinematic clustering, tabular and graph-model comparisons, and dashboard-based review. It does not cover real-time vehicle control, camera or LiDAR perception, or validated crash prediction.

**Approach.** The pipeline streams TFRecord scenes, removes implausible-speed observations, assigns a reproducible kinematic-complexity target, engineers spatial interaction metrics, and compares baseline and tuned models with a stratified holdout evaluation.

[Back to contents](#table-of-contents)

<a id="data-preparation"></a>
## 5. Data Preparation and Physical Anomaly Screening
Scenarios with a peak speed above 38.0 m/s (~85 mph) are excluded before modeling.

In [ ]:
# What: Load raw Waymo scenario TFRecords from Google Cloud Storage using configuration settings.
# Expected outcome: Extract df_raw scenarios and corresponding graph scene structures into memory.
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from src.config import GCS_BUCKET_PATH, MAX_VELOCITY_THRESHOLD, NUM_SHARDS, TOTAL_TRAINING_SHARDS
from src.data_loader import load_and_extract_waymo_scenarios

df_raw, gnn_extracted_scenes = load_and_extract_waymo_scenarios(
    GCS_BUCKET_PATH,
    num_shards=NUM_SHARDS,
    total_shards=TOTAL_TRAINING_SHARDS,
    velocity_threshold_mps=MAX_VELOCITY_THRESHOLD,
)
print(f'Extracted {len(df_raw):,} scenarios.')

### 5.1 Spatial Feature Engineering and Risk Target
The target follows the original four-cluster kinematic risk definition. Spatial features are joined by `scenario_id`, so filtering invalid telemetry cannot shift features onto the wrong scene.

In [ ]:
# What: Clean raw motion data, calculate persistent kinematic risk targets, and extract spatial features.
# Expected outcome: Return cleaned df_clean dataframe with risk targets mapped to target vector y.
import pandas as pd
from src.feature_engineering import DataCleaner, SpatialFeatureEngineer, assign_kinematic_risk_target

df_clean = DataCleaner(MAX_VELOCITY_THRESHOLD).clean_motion_tracks(df_raw)
df_clean = assign_kinematic_risk_target(df_clean)
df_clean = SpatialFeatureEngineer().transform(df_clean, gnn_extracted_scenes)
y = df_clean['target_risk_matrix'].map({'Standard Complexity': 0, 'Critical Complexity': 1})

display(df_clean[['max_velocity_mps', 'max_deceleration', 'min_inter_agent_dist', 'velocity_std', 'target_risk_matrix']].describe(include='all'))
print(y.value_counts().rename({0: 'Standard', 1: 'Critical'}))

### 5.2 Exploratory Quality and Risk Visualizations
These compact plots preserve the evidence shown in the original notebook without repeating its extraction implementation.

In [ ]:
# What: Generate data quality metrics table and distribution plots for velocity, vehicle count, and risk tiers.
# Expected outcome: Summary table displayed alongside 3 exploratory diagnostic charts.
import matplotlib.pyplot as plt
import seaborn as sns

quality_summary = pd.DataFrame({
    'Metric': ['Raw scenarios', 'Clean scenarios', 'Invalid physics scenarios', 'Missing cells'],
    'Value': [len(df_raw), len(df_clean), int((~df_raw['is_valid_physics']).sum()), int(df_raw.isna().sum().sum())],
})
display(quality_summary)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(df_raw['max_velocity_mps'], bins=40, kde=True, ax=axes[0], color='#E76F51')
axes[0].axvline(MAX_VELOCITY_THRESHOLD, color='black', linestyle='--', label='38 m/s threshold')
axes[0].set_title('Peak velocity and physics threshold')
axes[0].legend()

sns.boxplot(x=df_raw['vehicle_count'], ax=axes[1], color='#90BE6D')
axes[1].set_title('Vehicle-count distribution')

sns.countplot(data=df_clean, x='target_risk_matrix', hue='target_risk_matrix', legend=False, ax=axes[2], palette=['#4C72B0', '#DD8452'])
axes[2].set_title('Kinematic risk classes')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

[Back to contents](#table-of-contents)

<a id="risk-modeling"></a>
## 6. Risk Modeling and Fine-Tuning
The baseline uses static map friction and actor counts; the expanded models add spatial interaction metrics.

In [ ]:
# What: Perform stratified train-test split and train baseline tabular models (Random Forest, XGBoost, LightGBM).
# Expected outcome: Printed baseline test accuracy and ROC-AUC metrics for the Random Forest model.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from src.config import EXPANDED_FEATURES, RANDOM_STATE, TEST_SIZE
from src.modeling import initialize_tabular_models

X = df_clean[EXPANDED_FEATURES]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
xgb_model, lgb_model = initialize_tabular_models(X_train, y_train, X_test, y_test)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
print(f'Random Forest accuracy: {accuracy_score(y_test, rf_predictions):.4f}')
print(f'Random Forest ROC-AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.4f}')

### 6.1 Graph Attention Architecture
The reusable architecture takes the seven node features used in the original graph pipeline: actor type, velocity, roadgraph count, and four map-friction counts.

In [ ]:
# What: Instantiate the Graph Attention Network (GAT) PyTorch model architecture.
# Expected outcome: Active device confirmation (CPU or CUDA) and initialized GAT model instance.
import torch
from src.modeling import GraphAttentionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gat_model = GraphAttentionNet(in_channels=7).to(device)
print(f'Graph Attention Network initialized on {device}.')

### 6.2 Before-and-After Fine-Tuning
This section preserves the original analysis design: the same randomized-search grids tune Random Forest, XGBoost, and LightGBM by ROC-AUC. GCN tuning evaluates the four original dropout, width, learning-rate, and weight-decay settings. Run this only after the baseline cells; it can take several minutes.

In [ ]:
# What: Execute randomized hyperparameter tuning across Random Forest, XGBoost, and LightGBM models.
# Expected outcome: Displayed DataFrame comparing evaluation metrics before and after hyperparameter tuning.
from src.tuning import evaluate_classifier, tune_tabular_models

baseline_models = {'Random Forest': rf_model, 'XGBoost': xgb_model, 'LightGBM': lgb_model}
before_tuning = {name: evaluate_classifier(model, X_test, y_test) for name, model in baseline_models.items()}
tuned_models = tune_tabular_models(X_train, y_train)
after_tuning = {name: evaluate_classifier(model, X_test, y_test) for name, model in tuned_models.items()}

comparison = pd.concat([
    pd.DataFrame(before_tuning).T.assign(Stage='Before tuning'),
    pd.DataFrame(after_tuning).T.assign(Stage='After tuning'),
]).reset_index(names='Model Family')
display(comparison[['Stage', 'Model Family', 'Test Acc', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].sort_values(['Stage', 'Model Family']))

In [ ]:
# What: Render bar chart comparing held-out test accuracy before and after hyperparameter tuning.
# Expected outcome: Grouped bar chart visualization showing relative tuning performance gains.
plt.figure(figsize=(10, 4))
sns.barplot(data=comparison, x='Model Family', y='Test Acc', hue='Stage', palette='Set2')
plt.ylim(0, 1)
plt.ylabel('Held-out test accuracy')
plt.xlabel('')
plt.title('Baseline versus tuned model performance')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

### 6.3 Explainable AI Evidence
The SHAP summary below connects the selected tree-model results to the white paper's interpretation of motion variance, proximity, and map context.

In [ ]:
# What: Calculate TreeExplainer SHAP values to quantify global feature impact on scenario risk scoring.
# Expected outcome: Rendered SHAP summary plot ordering features by average impact magnitude.
import shap

shap_explainer = shap.TreeExplainer(xgb_model)
shap_values = shap_explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, feature_names=X_test.columns, show=False)
plt.title('Global SHAP feature importance')
plt.tight_layout()
plt.show()

### Tuned Spatial GCN
This reproduces the original baseline GCN and its four-setting tuned counterpart. The held-out graph split is fixed with the same random seed as the tabular comparison.

In [ ]:
# What: Construct PyTorch Geometric scene graphs, train baseline SpatialGCN, and tune GCN hyperparameters.
# Expected outcome: Printed metrics table comparing baseline vs tuned SpatialGCN and best hyperparameter choices.
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from torch_geometric.loader import DataLoader
from src.modeling import SpatialGCN, build_scene_graphs
from src.tuning import tune_gcn

graph_dataset = build_scene_graphs(df_clean, gnn_extracted_scenes, y)
graph_targets = [graph.y.item() for graph in graph_dataset]
train_graphs, test_graphs = train_test_split(graph_dataset, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=graph_targets)
train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

baseline_gcn = SpatialGCN(in_channels=7).to(device)
optimizer = torch.optim.Adam(baseline_gcn.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()
baseline_gcn.train()
for _ in range(30):
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        loss = criterion(baseline_gcn(batch.x, batch.edge_index, batch.batch), batch.y)
        loss.backward()
        optimizer.step()

def score_gcn(model):
    model.eval()
    targets, predictions, probabilities = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            output = model(batch.x, batch.edge_index, batch.batch)
            targets.extend(batch.y.cpu().numpy())
            predictions.extend(output.argmax(dim=1).cpu().numpy())
            probabilities.extend(F.softmax(output, dim=1)[:, 1].cpu().numpy())
    return {
        'Test Acc': accuracy_score(targets, predictions),
        'Precision': precision_score(targets, predictions, zero_division=0),
        'Recall': recall_score(targets, predictions, zero_division=0),
        'F1-Score': f1_score(targets, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(targets, probabilities)
    }

gcn_before = score_gcn(baseline_gcn)
best_gcn = tune_gcn(train_loader, test_loader, device)
gcn_comparison = pd.DataFrame([{'Stage': 'Before tuning', **gcn_before}, {'Stage': 'After tuning', **best_gcn['metrics']}])
display(gcn_comparison)
print(f"Best GCN parameters: {best_gcn['params']}")

[Back to contents](#table-of-contents)

<a id="limitations"></a>
## 7. Limitations
The risk label is derived from kinematic clustering rather than verified crash or near-miss outcomes, so it represents scene complexity rather than a calibrated probability of collision. The analytical sample is limited to the selected Waymo scenarios and aggregate 20-second summaries; it does not establish performance in severe weather, at night, or in every geographic setting. Graph-model results should also be interpreted as a benchmark until they are evaluated with a fully independent operational test set.

[Back to contents](#table-of-contents)

<a id="solution-details"></a>
## 8. Solution Details
The operational deliverable is an interactive Streamlit safety-triage dashboard. It exposes the model's scenario-risk score and supports human review of the supporting motion and map context. This makes the analysis useful beyond a one-time notebook: reviewers can inspect flagged scenarios, assess the evidence, and use their judgment when prioritizing follow-up work.

**Live application:** [Waymo Scene Safety Triage Dashboard](https://waymo-scene-safety-triage.streamlit.app/)

[Back to contents](#table-of-contents)

<a id="concluding-summary"></a>
## 9. Concluding Summary and Call to Action
This project demonstrates a practical trade-off: transparent, engineered features and lightweight models can make large-scale scenario review faster and more explainable, while graph architectures remain a useful accuracy benchmark. The next step is to use the dashboard to inspect high-risk scenes, record reviewer feedback, and prioritize a future evaluation against independently labeled safety events.

**Call to action:** Open the [Streamlit dashboard](https://waymo-scene-safety-triage.streamlit.app/) to explore the triage workflow, then review the repository's model and data pipeline for reproducibility.

---
### Acknowledgements & Tooling Disclosure
* **AI Assistance:** Google Gemini was utilized as a supportive assistant for code linting (PEP 8 adherence), notebook header comment standardization, and cell navigation hotlink formatting. All core analytical logic, feature engineering designs, and model selection decisions remain the original work of the author.

[Back to contents](#table-of-contents)